# Load or Restore the Synthetic E-Commerce Snapshot to BigQuery

Use this notebook for both the **initial BigQuery load** and future **restores**
after BigQuery Sandbox tables expire.

This notebook:

- does not run the synthetic-data generator;
- does not create new random values;
- reads the 13 Parquet files from `synthetic_ecommerce.zip`;
- recreates the exact snapshot as standard, non-partitioned BigQuery tables;
- verifies every uploaded table against the source row count.

BigQuery Sandbox automatically expires tables, views, and partitions after
approximately 60 days. The Parquet archive therefore remains the permanent
source of truth.

Official reference: https://cloud.google.com/bigquery/docs/sandbox

> Warning: the upload step replaces only the 13 named synthetic tables in the
> configured destination dataset.

## 1. Install dependencies

Run this cell once at the beginning of a new Colab session.

In [ ]:
# Colab setup
%pip -q install pyarrow google-cloud-bigquery

## 2. Configuration

Edit only the values marked `YOUR_...`.

- `DRIVE_FOLDER`: folder containing `synthetic_ecommerce.zip`
- `GCP_PROJECT`: Google Cloud Project ID
- `BQ_DATASET`: destination BigQuery dataset
- `BQ_LOCATION`: BigQuery dataset location

The loader always creates non-partitioned tables to remain safe for historical
2024 data in BigQuery Sandbox.

In [ ]:
from pathlib import Path
import shutil
import zipfile

import pandas as pd

# ---------- Edit these settings ----------
DRIVE_FOLDER = Path("/content/drive/MyDrive/YOUR_FOLDER")
PARQUET_ZIP_NAME = "synthetic_ecommerce.zip"

GCP_PROJECT = "YOUR_GCP_PROJECT_ID"
BQ_DATASET = "synthetic_ecommerce_portfolio"
BQ_LOCATION = "asia-southeast2"

# Optional: create a CSV ZIP beside the Parquet ZIP.
CREATE_CSV_BACKUP = False
CSV_BACKUP_NAME = "synthetic_ecommerce_csv.zip"
# -----------------------------------------

PARQUET_ZIP = DRIVE_FOLDER / PARQUET_ZIP_NAME

EXPECTED_TABLES = [
    "dim_date",
    "dim_users",
    "dim_merchants",
    "dim_products",
    "dim_campaigns",
    "bridge_campaign_skus",
    "fact_sessions",
    "fact_funnel_events",
    "fact_orders",
    "fact_order_items",
    "fact_inventory_daily",
    "fact_returns",
    "data_dictionary"
]

## 3. Read and validate the archived snapshot

The ZIP is extracted into temporary Colab storage. The notebook stops before
upload if a required table is missing or empty.

In [ ]:
from google.colab import drive

if "YOUR_" in str(DRIVE_FOLDER):
    raise ValueError("Update DRIVE_FOLDER in the configuration cell first.")

drive.mount("/content/drive")

if not PARQUET_ZIP.exists():
    raise FileNotFoundError(
        f"Snapshot ZIP not found: {PARQUET_ZIP}\n"
        "Check DRIVE_FOLDER and PARQUET_ZIP_NAME."
    )

extract_dir = Path("/content/synthetic_ecommerce_snapshot")
if extract_dir.exists():
    shutil.rmtree(extract_dir)
extract_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(PARQUET_ZIP, "r") as archive:
    archive.extractall(extract_dir)

parquet_files = sorted(extract_dir.rglob("*.parquet"))
tables = {
    file_path.stem: pd.read_parquet(file_path)
    for file_path in parquet_files
}

found_tables = set(tables)
expected_tables = set(EXPECTED_TABLES)
missing_tables = expected_tables - found_tables
unexpected_tables = found_tables - expected_tables

if missing_tables:
    raise ValueError(
        f"Snapshot is incomplete. Missing tables: {sorted(missing_tables)}"
    )

if unexpected_tables:
    print("Warning — unexpected Parquet files:", sorted(unexpected_tables))

empty_tables = sorted(
    table_name
    for table_name, dataframe in tables.items()
    if dataframe.empty
)
if empty_tables:
    raise ValueError(
        f"Upload cancelled. Empty source tables: {empty_tables}"
    )

source_summary = pd.DataFrame([
    {
        "table_name": table_name,
        "rows": len(dataframe),
        "columns": len(dataframe.columns),
    }
    for table_name, dataframe in tables.items()
]).sort_values("table_name").reset_index(drop=True)

print(f"Validated {len(tables)} Parquet tables.")
display(source_summary)

## 4. Optional CSV backup

Parquet is the primary backup because it preserves data types and is more
compact. Set `CREATE_CSV_BACKUP = True` only if a conventional CSV copy is
needed.

In [ ]:
if CREATE_CSV_BACKUP:
    csv_dir = Path("/content/synthetic_ecommerce_csv")
    if csv_dir.exists():
        shutil.rmtree(csv_dir)
    csv_dir.mkdir(parents=True, exist_ok=True)

    for table_name, dataframe in tables.items():
        dataframe.to_csv(
            csv_dir / f"{table_name}.csv",
            index=False,
        )

    local_csv_archive = shutil.make_archive(
        "/content/synthetic_ecommerce_csv",
        "zip",
        csv_dir,
    )
    destination = DRIVE_FOLDER / CSV_BACKUP_NAME
    shutil.copy2(local_csv_archive, destination)
    print("CSV backup saved:", destination)
else:
    print("CSV backup skipped.")

## 5. Upload to BigQuery as non-partitioned tables

The cell authenticates with Google, creates the dataset when needed, and then
replaces exactly the 13 expected tables.

No date partitioning or clustering is applied. This keeps the loader simple and
prevents historical 2024 partitions from disappearing immediately in Sandbox.

In [ ]:
from google.colab import auth
from google.cloud import bigquery

if GCP_PROJECT == "YOUR_GCP_PROJECT_ID":
    raise ValueError("Update GCP_PROJECT in the configuration cell first.")

auth.authenticate_user()
client = bigquery.Client(project=GCP_PROJECT)

dataset_id = f"{GCP_PROJECT}.{BQ_DATASET}"
dataset = bigquery.Dataset(dataset_id)
dataset.location = BQ_LOCATION
client.create_dataset(dataset, exists_ok=True)

for table_name in EXPECTED_TABLES:
    dataframe = tables[table_name]
    table_id = f"{dataset_id}.{table_name}"

    # Delete the old table so a previous partition configuration is not retained.
    client.delete_table(table_id, not_found_ok=True)

    job_config = bigquery.LoadJobConfig(
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
    )

    load_job = client.load_table_from_dataframe(
        dataframe,
        table_id,
        job_config=job_config,
    )
    load_job.result()

    loaded_table = client.get_table(table_id)
    print(
        f"Loaded {table_name}: "
        f"{loaded_table.num_rows:,} rows | "
        f"partitioned={loaded_table.time_partitioning is not None}"
    )

print("All 13 tables were loaded into BigQuery.")

## 6. Verify the BigQuery upload

The final check confirms that:

- every source row count matches BigQuery;
- every table is non-partitioned;
- all 13 expected tables were loaded.

In [ ]:
verification = []

for table_name in EXPECTED_TABLES:
    source_dataframe = tables[table_name]
    table_id = f"{GCP_PROJECT}.{BQ_DATASET}.{table_name}"
    destination_table = client.get_table(table_id)

    verification.append({
        "table_name": table_name,
        "source_rows": len(source_dataframe),
        "bigquery_rows": destination_table.num_rows,
        "row_count_match": (
            len(source_dataframe) == destination_table.num_rows
        ),
        "partitioned": (
            destination_table.time_partitioning is not None
        ),
    })

verification_df = (
    pd.DataFrame(verification)
    .sort_values("table_name")
    .reset_index(drop=True)
)
display(verification_df)

if not verification_df["row_count_match"].all():
    raise ValueError(
        "Row-count verification failed. "
        "Review rows with row_count_match=False."
    )

if verification_df["partitioned"].any():
    raise ValueError(
        "At least one destination table is still partitioned."
    )

print("Verification passed: all row counts match.")
print("Verification passed: all destination tables are non-partitioned.")

## Usage summary

1. Generate the dataset once with `01_generate_synthetic_data.ipynb`.
2. Keep `synthetic_ecommerce.zip` as the permanent snapshot.
3. Use this notebook for the initial BigQuery load.
4. Run this notebook again when Sandbox tables expire.
5. Do not rerun the generator when the goal is to preserve the portfolio's
   exact numbers.

The BigQuery environment is temporary; the archived Parquet snapshot is the
reproducible source of truth.